# Human Genome Diversity Project (HGDP)

Compare the Haplosaurus haplotypes from the [1000 Genomes Project - Phase 3 (1KG)](https://www.internationalgenome.org/data-portal/data-collection/grch38) vs. the [Human Genome Diversity Project (HGDP)](https://www.internationalgenome.org/data-portal/data-collection/hgdp)

In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import os
# only load this one time per session
if 'NOTEBOOK_INITIALIZED' not in globals():
    os.chdir(os.path.dirname(os.path.abspath('.')))
    NOTEBOOK_INITIALIZED = True

import src.utils as utils
import src.config as config
import src.haplosaurus as hs
import src.ESM as ESM
import src.ESM_predict as ESMp
import src.gprofiler as gp
import src.vep_pipeline as vp
import src.vep_analysis as va
import src.vep_metrics as vm
import src.proteingym as pg 
import src.ensembl_rest as er
import src.biopython as bp
import src.onekg as og

pd.set_option('display.max_columns', None)

/home/schilder/.conda/envs/esm2/lib/python3.12/site-packages/Bio/Application/__init__.py:39: BiopythonDeprecationWarning: The Bio.Application modules and modules relying on it have been deprecated.

Due to the on going maintenance burden of keeping command line application
wrappers up to date, we have decided to deprecate and eventually remove these
modules.

We instead now recommend building your command line and invoking it directly
with the subprocess module.
  warnings.warn(


In [ ]:
%%capture

haplotypes_save_dir = hs.split_haplosaurus_results(merged_json="../data/haplosaurus/hgdp_autosomes_output.json.gz",
                              # skip excessively long lines which slows down the function significantly
                              max_line_len=1e6
                              )
haplotypes_save_dir

## Compare transcripts in HGDP, 1kG, and ProteinGym

In [2]:
pg_df = pg.merge_resources(keys = ['clinical_ProteinGym_substitutions.zip']) 

In [4]:
tx_ids_1kg = hs.list_haplotypes(cache = hs.DIR_DICT["haplotypes"])
tx_ids_hgdp = hs.list_haplotypes(cache = hs.DIR_DICT["HGDP_haplotypes"])

# Get the tx_ids that are in 1KG but not in HGDP
tx_ids_1kg_not_hgdp = set(tx_ids_1kg) - set(tx_ids_hgdp)
print(len(tx_ids_1kg_not_hgdp), "tx_ids in 1KG but not in HGDP")

# Get the tx_ids that are in HGDP but not in 1KG
tx_ids_hgdp_not_1kg = set(tx_ids_hgdp) - set(tx_ids_1kg)
print(len(tx_ids_hgdp_not_1kg), "tx_ids in HGDP but not in 1KG")

# Get the tx_ids that are in both 1KG and HGDP
tx_ids_both = set(tx_ids_1kg) & set(tx_ids_hgdp)
print(len(tx_ids_both), "tx_ids in both 1KG and HGDP")

# Get the tx_ids that are in ProteinGym and 1KG and HGDP
tx_ids_pg = set(pg_df['ENST'].unique())
tx_ids_all = set(tx_ids_both) & set(tx_ids_pg)
print(len(tx_ids_all), "tx_ids in ProteinGym and 1KG and HGDP")


Found haplotypes of 47325 transcripts in: '/home/schilder/.cache/ensembl_rest/haplotypes/'
Found haplotypes of 85520 transcripts in: '/home/schilder/projects/data/Human_Genome_Diversity_Project/haplosaurus/'
7343 tx_ids in 1KG but not in HGDP
45538 tx_ids in HGDP but not in 1KG
39982 tx_ids in both 1KG and HGDP
2304 tx_ids in ProteinGym and 1KG and HGDP


## Compare haplotypes in HGDP, 1KG, and ProteinGym

In [5]:
haplotypes_hgdp = hs.get_haplotypes(cache = hs.DIR_DICT["HGDP_haplotypes"],
                                    tx_ids = tx_ids_all,
                                    add_missing_ref=True,
                                    cache_only = True)

Found haplotypes of 85520 transcripts in: '/home/schilder/projects/data/Human_Genome_Diversity_Project/haplosaurus/'


Getting haplotypes:   0%|          | 0/2304 [00:00<?, ?it/s]

Adding reference haplotype:   0%|          | 0/2304 [00:00<?, ?it/s]

Adding reference haplotype for ENST00000217133
Adding reference haplotype for ENST00000263035
Adding reference haplotype for ENST00000269844
Adding reference haplotype for ENST00000283131
Adding reference haplotype for ENST00000301365
Adding reference haplotype for ENST00000301365
Adding reference haplotype for ENST00000344683
Adding reference haplotype for ENST00000344683
Adding reference haplotype for ENST00000354386
Adding reference haplotype for ENST00000354386
Adding reference haplotype for ENST00000359396
Adding reference haplotype for ENST00000360467
Adding reference haplotype for ENST00000367674
Adding reference haplotype for ENST00000370225
Adding reference haplotype for ENST00000371589
Adding reference haplotype for ENST00000373631
Adding reference haplotype for ENST00000373631
Adding reference haplotype for ENST00000376970
Adding reference haplotype for ENST00000389934
Adding reference haplotype for ENST00000403437
Adding reference haplotype for ENST00000407559
Adding refere

In [6]:
haplotypes_1kg = hs.get_haplotypes(cache = hs.DIR_DICT["haplotypes"],
                                    tx_ids = tx_ids_all,
                                    add_missing_ref=True,
                                    cache_only = True)

Found haplotypes of 47325 transcripts in: '/home/schilder/.cache/ensembl_rest/haplotypes/'


Getting haplotypes:   0%|          | 0/2304 [00:00<?, ?it/s]

Adding reference haplotype:   0%|          | 0/2304 [00:00<?, ?it/s]

Adding reference haplotype for ENST00000241041
Adding reference haplotype for ENST00000241041
Adding reference haplotype for ENST00000269844
Adding reference haplotype for ENST00000283131
Adding reference haplotype for ENST00000285021
Adding reference haplotype for ENST00000302850
Adding reference haplotype for ENST00000302850
Adding reference haplotype for ENST00000303236
Adding reference haplotype for ENST00000303236
Adding reference haplotype for ENST00000319023
Adding reference haplotype for ENST00000319023
Adding reference haplotype for ENST00000341500
Adding reference haplotype for ENST00000341500
Adding reference haplotype for ENST00000373578
Adding reference haplotype for ENST00000376970
Adding reference haplotype for ENST00000421182
Adding reference haplotype for ENST00000421182
Adding reference haplotype for ENST00000441037


In [7]:
df_hgdp = hs.haplotypes_to_df(haplotypes=haplotypes_hgdp, add_consensus=False)
df_1kg = hs.haplotypes_to_df(haplotypes=haplotypes_1kg, add_consensus=False) 

Adding reference haplotype:   0%|          | 0/2304 [00:00<?, ?it/s]

Getting haplotype sequences:   0%|          | 0/2304 [00:00<?, ?it/s]

Getting haplotype names:   0%|          | 0/2304 [00:00<?, ?it/s]

Converting haplotypes to dataframe:   0%|          | 0/2304 [00:00<?, ?it/s]

Adding reference haplotype:   0%|          | 0/2304 [00:00<?, ?it/s]

Getting haplotype sequences:   0%|          | 0/2304 [00:00<?, ?it/s]

Getting haplotype names:   0%|          | 0/2304 [00:00<?, ?it/s]

Converting haplotypes to dataframe:   0%|          | 0/2304 [00:00<?, ?it/s]

In [8]:
hap_ids_hgdp = set(df_hgdp.index)
hap_ids_1kg = set(df_1kg.index)

# Haplotype IDs in HGDP but not in 1KG
hap_ids_hgdp_not_1kg = hap_ids_hgdp - hap_ids_1kg
print(len(hap_ids_hgdp_not_1kg), "haplotype IDs in HGDP but not in 1KG")

# Haplotype IDs in 1KG but not in HGDP
hap_ids_1kg_not_hgdp = hap_ids_1kg - hap_ids_hgdp
print(len(hap_ids_1kg_not_hgdp), "haplotype IDs in 1KG but not in HGDP")

# Haplotype IDs in both 1KG and HGDP
hap_ids_both = hap_ids_hgdp & hap_ids_1kg
print(len(hap_ids_both), "haplotype IDs in both 1KG and HGDP")


# Get the union of haplotype IDs in HGDP and 1KG
hap_ids_union = hap_ids_hgdp | hap_ids_1kg
print(len(hap_ids_union), "haplotype IDs in HGDP or 1KG")
 

27127 haplotype IDs in HGDP but not in 1KG
71945 haplotype IDs in 1KG but not in HGDP
23715 haplotype IDs in both 1KG and HGDP
122787 haplotype IDs in HGDP or 1KG


## Find out which samples we are missing from our haplotype collections

Merge all haplotypes into one dictionary.

In [11]:
import warnings

# Create a list to store warning messages
merge_warnings = []

# Define a custom warning handler
def warning_collector(message, category, filename, lineno, file=None, line=None):
    merge_warnings.append(str(message))

# Use the custom warning handler in a context manager
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    # Temporarily override showwarning to collect warnings
    original_showwarning = warnings.showwarning
    warnings.showwarning = warning_collector

    haplotypes = hs.merge_haplotype_datasets(
        {"1KG": haplotypes_1kg, "HGDP": haplotypes_hgdp},
        tx_id_method="intersection",
        protein_key_missing="warning",
        cds_key_missing=None,
        use_deepcopy=False
    )

    # Restore the original showwarning function
    warnings.showwarning = original_showwarning

all_samples = set(hs.get_haplotype_samples(haplotypes, unnest=True))
print(f"Samples collected so far: {len(all_samples)}")
# Now merge_warnings contains all warning messages generated during merging

Using copy
Merging 2304 tx_ids using intersection


Merging haplotype datasets: 0it [00:00, ?it/s]

Merging haplotypes:   0%|          | 0/2304 [00:00<?, ?it/s]

Getting haplotype names:   0%|          | 0/2304 [00:00<?, ?it/s]

Merged 122787 haplotypes across 2304 tx_ids.


Extracting sample IDs:   0%|          | 0/2304 [00:00<?, ?it/s]

Samples collected so far: 3938


These are the transcripts that are missing the REF sequence information about sample maps and frequencies (needed to reconstruct individuals).

In [14]:
set([x.split(" ")[1] for x in merge_warnings])

{'ENST00000301365',
 'ENST00000344683',
 'ENST00000354386',
 'ENST00000359396',
 'ENST00000373631',
 'ENST00000414423',
 'ENST00000576742'}

In [42]:
og_metadata = og.get_sample_metadata()

og_missing = og_metadata.loc[~og_metadata["sample"].isin(list(all_samples))]
og_missing["collection"] = og_missing["Data collections"].str.split(",")
og_missing = og_missing.explode("collection")
print("Number of samples missing:",og_missing["sample"].nunique())
og_missing.groupby("collection")["sample"].nunique().sort_values(ascending=False) 

Number of samples missing: 4067


/tmp/ipykernel_2632295/642933094.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  og_missing["collection"] = og_missing["Data collections"].str.split(",")


collection
1000 Genomes 30x on GRCh38                          3202
1000 Genomes phase 3 release                        3115
1000 Genomes on GRCh38                              2709
1000 Genomes phase 1 release                        1182
1KG_ONT_VIENNA                                      1019
MAGE RNA-seq                                         731
Gambian Genome Variation Project (GRCh38)            518
Geuvadis                                             465
Gambian Genome Variation Project (GRCh37)            400
Simons Genome Diversity Project                      145
90 Han Chinese high coverage genomes                  90
Human Genome Structural Variation Consortium          70
 Phase 3                                              67
 Phase 2                                              44
Human Genome Diversity Project                        26
The Human Genome Structural Variation Consortium       9
Illumina Platinum pedigree                             6
Name: sample, dtype: